<a href="https://colab.research.google.com/github/matvey-gostev/Calculation-Of-Absolute-Permeability-Of-Rock-Based-On-Topological-Features/blob/main/main_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Структура обработанного датасета DRP-372 (только образцы 256x256x256)

#### Папки

- data/npy/ - бинарные 3D-структуры пород (файлы *.npy)
- data/csv/ - целевые значения проницаемости (файлы *.csv)

#### Формат .npy (входные данные)

- Размерность = (256, 256, 256)
- Тип данных: float32 (0 - поровое пространство, 1 - твёрдая фаза)
- Загрузка: `vol = np.load("data/npy/204_03_256.npy")`

#### Формат .csv (целевые переменные)

Каждый CSV в data/csv/ содержит 6 строк (давления 1,2,5,10,20 МПа и LBM):

| pressure | permeability |
|----------|--------------|
| 1        |       |
| 2        |       |
| 5        |       |
| 10       |       |
| 20       |      |
| LBM      |     |

- pressure: давление в МПа (для LBM - строка 'LBM')
- permeability: проницаемость (при отсутствии данных - NaN)

Также все csv объединены в один файл с именем "dataset_combined.csv", расположенный в корне проекта

#### PyTorch Dataset

Класс PermDataset возвращает пару (volume, target_vector):
- volume: тензор (1, 256, 256, 256)
- target: тензор (6,)

```python
dataset = PermDataset()
vol, perm = dataset[0]  # vol.shape=(1,256,256,256), perm.shape=(6,)

## Все необходимые загрузки библиотек и импорты

In [ ]:
!pip install "numpy<2.0" "scikit-learn<1.5" ripser persim
import numpy as np
print(f"Current NumPy version: {np.__version__}")
if np.__version__.startswith('2.'):
    print("\n[!] ВНИМАНИЕ: Нужно перезапустить сессию (Runtime -> Restart session), чтобы изменения вступили в силу.")

np.set_printoptions(precision=2, suppress=True, linewidth=120)

In [ ]:
!pip install pyvista numpy
!pip install gudhi
!pip install giotto-tda
!pip install ripser
!pip install -U git+https://github.com/shizuo-kaji/CubicalRipser_3dim
#!pip install remotezip

In [ ]:
#from remotezip import RemoteZip
#import h5py

import pandas as pd
from tqdm.notebook import tqdm
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import Adam
from torch.nn import CrossEntropyLoss

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import ListedColormap

from sklearn.datasets import make_circles
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.decomposition import PCA
from sklearn.manifold import Isomap

import scipy as sp
from scipy import stats
from scipy.ndimage import zoom

import fnmatch
import gdown
import zipfile
import io
import glob
import os

# gudhi
import gudhi as gd
from gudhi import bottleneck_distance
from gudhi.hera import wasserstein_distance
from gudhi.representations import BettiCurve, PersistenceImage, Landscape

# giotto
import gtda
from gtda.homology import VietorisRipsPersistence
# from gtda.diagrams import BettiCurve, PersistenceImage, PersistenceLandscape
# Для работы с кубикальными комплексами (изображениями)
from gtda.homology import CubicalPersistence
# Для визуализации диаграмм
from gtda.plotting import plot_diagram
# Для подготовки данных (бинаризация, фильтрация)
from gtda.images import Binarizer, HeightFiltration
import gtda.plotting as gtdaplot

# ripser
from ripser import lower_star_img
from ripser import Rips


import pyvista as pv
from IPython.display import Image, display
from persim import plot_diagrams
import pickle
import cripser

## Получение и объединение .csv файлов с результатами симуляций, преобразование .mat -> .npy

####Код подгрузки данных первого датасета на случай потери доступа к гугл диску

In [ ]:
# @title


# Код реорганизации данных
'''
url = "https://web.corral.tacc.utexas.edu/digitalporousmedia/archive/DRP-372/DRP-372_archive.zip"
os.makedirs("data/npy", exist_ok=True)
os.makedirs("data/csv", exist_ok=True)

def cb(name, obj):
  if isinstance(obj, h5py.Dataset) and obj.ndim == 3:
      return obj

def find_3d_dataset(h5_file):
  """Возвращает 3D массив или None."""
  ds = h5_file.visititems(cb)
  return ds

def get_permeability(csv_bytes):
  """Извлекает значение permeability из CSV."""
  try:
      text = csv_bytes.decode('utf-8')
      for line in text.splitlines():
          if 'permeability' in line.lower():
              parts = line.split(',')
              if len(parts) >= 2:
                  val = parts[1].strip()
                  if not val and len(parts) > 0:
                      val = parts[0].strip()
                  return float(val)
      return None
  except Exception as e:
      print(f"Ошибка парсинга: {e}")
      return None

with RemoteZip(url) as rz:
    all_files = rz.namelist()
    mat_files = [f for f in all_files if '_256.mat' in f and f.endswith('.mat')]

    for mat_path in tqdm(mat_files):
        name = mat_path.split('/')[-1][:-4]
        base_dir = '/'.join(mat_path.split('/')[:-1])

        #.mat -> .npy
        npy_path = f"data/npy/{name}.npy"
        if not os.path.exists(npy_path):
            try:
                rz.extract(mat_path, path='.')
                with h5py.File(mat_path, 'r') as hf:
                    vol = find_3d_dataset(hf)
                    if vol is not None:
                        np.save(npy_path, vol)
                os.remove(mat_path)
            except Exception as e:
                print(f"Ошибка при обработке {name}: {e}")
                continue

        # Поиск permeability
        perms = []
        pressures = [1,2,5,10,20,'LBM']
        for p in pressures:
            if p == 'LBM':
                pattern = f"{base_dir}/Single Phase*/LBM.csv"
            else:
                pattern = f"{base_dir}/Single Phase*/P_{p}_MPa.csv"
            matching = fnmatch.filter(all_files, pattern)
            if matching:
                csv_path = matching[0]
                try:
                    content = rz.read(csv_path)
                    perm = get_permeability(content)
                    perms.append(perm)
                except Exception as e:
                    print(f"Не удалось прочитать {csv_path}: {e}")
                    perms.append(None)
            else:
                perms.append(None)

        df_out = pd.DataFrame({'pressure': pressures, 'permeability': perms})
        df_out.to_csv(f"data/csv/{name}.csv", index=False)
        print(f"{name}: {perms}")
'''

####Подгрузка первого датасета из гугл диска

In [ ]:
ZIP_URL = "https://drive.google.com/uc?export=download&id=1R40Kau_i1qEG3ge0by_sFncIXz7d0rCw"
ZIP_PATH = "data.zip"
EXTRACT_DIR = "data"

if not os.path.exists(EXTRACT_DIR):
    print("Скачивание архива с данными...")
    gdown.download(ZIP_URL, ZIP_PATH, quiet=False)

    print("Распаковка архива...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)

    os.remove(ZIP_PATH)
    print("Готово! Данные загружены в папку 'data'.")
else:
    print("Папка 'data' уже существует, пропускаем загрузку.")

#### Подгрузим файл с именами горных пород

In [ ]:
url = "https://docs.google.com/spreadsheets/d/1toagV4m4SN2T5RBMHSssU9RVSO4Zf9us/export?format=xlsx"
output = "metadata.xlsx"
gdown.download(url, output, quiet=False)

#### Подгрузка второго датасета (1000 вхождений), опционально

In [ ]:
FILE_ID = "1uDh0Vxdo-xHBgdAmI0h_DjFGDkMCKoVa"
ZIP_PATH = "synthetic_data_256.zip"
EXTRACT_DIR = "synthetic_data_extracted"

def download_synthetic_zip():
    if not os.path.exists(ZIP_PATH):
        print("Скачиваем архив с Google Диска...")
        gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", ZIP_PATH, quiet=False)
    else:
        print("Архив уже скачан.")

def extract_zip():
    if not os.path.exists(EXTRACT_DIR):
        print("Распаковываем архив...")
        with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
            zf.extractall(EXTRACT_DIR)
    else:
        print("Архив уже распакован.")

download_synthetic_zip()
extract_zip()

## Создание класса датасета, унаследованного от torch.utils.data.Dataset

In [ ]:
def normalize_subsample(s):
    parts = s.split('_')
    norm = []
    for p in parts:
        try:
            norm.append(str(int(p)))
        except ValueError:
            norm.append(p)
    return '_'.join(norm)

def parse_name_for_rock_type(name):
    parts = name.split('_')
    if len(parts) < 3:
        return None, None
    try:
        sample = int(parts[0])
        sub_raw = '_'.join(parts[1:-1])
        sub_norm = normalize_subsample(sub_raw)
        return sample, sub_norm
    except ValueError:
        return None, None

class PermeabilityDataset(Dataset):
    def __init__(self, real_npy_dir='data/npy', real_csv_dir='data/csv',
                 metadata_path='metadata.xlsx',
                 synthetic_extract_dir='synthetic_data_extracted',
                 use_synthetic=True):
        self.real_npy_dir = real_npy_dir
        self.real_csv_dir = real_csv_dir
        self.synth_dir = synthetic_extract_dir
        self.use_synthetic = use_synthetic

        meta_df = pd.read_excel(metadata_path, sheet_name='Table_SI')
        meta_df = meta_df[['Sample', 'Subsample', 'Type']].copy()
        meta_df['Sample'] = meta_df['Sample'].fillna(method='ffill')
        meta_df.dropna(subset=['Sample'], inplace=True)
        meta_df['Sample'] = meta_df['Sample'].astype(int)
        meta_df['Subsample'] = meta_df['Subsample'].astype(str)

        self.meta_dict = {}
        self.fallback_dict = {}
        for _, row in meta_df.iterrows():
            sample = row['Sample']
            norm_sub = normalize_subsample(row['Subsample'])
            key = (sample, norm_sub)
            if key not in self.meta_dict:
                self.meta_dict[key] = row['Type']
            if sample not in self.fallback_dict:
                self.fallback_dict[sample] = row['Type']

        self.samples = []

        real_names = [f[:-4] for f in os.listdir(real_npy_dir) if f.endswith('.npy')]
        real_names = [s for s in real_names if os.path.exists(os.path.join(real_csv_dir, f"{s}.csv"))]
        real_names = sorted(real_names)
        for idx, name in enumerate(real_names, start=1):
            self.samples.append({
                'type': 'real',
                'name': name,
                'ext_id': idx,   # внешний индекс
                'npy_path': os.path.join(real_npy_dir, f"{name}.npy"),
                'csv_path': os.path.join(real_csv_dir, f"{name}.csv")
            })

        if self.use_synthetic:
            synth_csv = os.path.join(self.synth_dir, 'synthetic_dataset.csv')
            synth_npy_dir = os.path.join(self.synth_dir, 'npy_256')
            if os.path.exists(synth_csv):
                df_synth = pd.read_csv(synth_csv)
                for _, row in df_synth.iterrows():
                    name = row['sample']
                    if name.endswith('.npy'):
                        name = name[:-4]
                    lbm_val = row['perm_LBM']
                    if pd.isna(lbm_val):
                        lbm_val = 0.0
                    self.samples.append({
                        'type': 'synth',
                        'name': name,
                        'ext_id': None,
                        'npy_path': os.path.join(synth_npy_dir, f"{name}.npy"),
                        'kappa': float(lbm_val)
                    })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        if sample['type'] == 'real':
            return self._load_real(sample)
        else:
            return self._load_synthetic(sample)

    def _load_real(self, sample):
        vol = np.load(sample['npy_path'])
        vol = torch.tensor(vol, dtype=torch.float32).unsqueeze(0)

        df = pd.read_csv(sample['csv_path'])
        perm = df['permeability'].values.astype(np.float32)
        perm = np.nan_to_num(perm, nan=0.0)
        perm = torch.tensor(perm, dtype=torch.float32)

        return vol, perm, sample['name']

    def _load_synthetic(self, sample):
        vol = np.load(sample['npy_path'])
        vol = torch.tensor(vol, dtype=torch.float32).unsqueeze(0)

        kappa_m2 = sample['kappa'] * 1e-12
        perm = torch.full((6,), 0.0, dtype=torch.float32)
        perm[5] = kappa_m2

        return vol, perm, sample['name']

    def get_rock_type(self, name):
        for s in self.samples:
            if s['name'] == name:
                if s['type'] == 'real':
                    sample, sub_norm = parse_name_for_rock_type(name)
                    if sample is None:
                        return "unknown"
                    key = (sample, sub_norm)
                    if key in self.meta_dict:
                        return self.meta_dict[key]
                    return self.fallback_dict.get(sample, "unknown")
                else:
                    return "synthetic"
        return "unknown"

    def get_name(self, idx):
        return self.samples[idx]['name']

    def remove_by_external_index(self, external_id):
        """Удаляет реальный образец по его внешнему индексу (как в dataset_combined.csv)."""
        idx_to_remove = None
        for i, s in enumerate(self.samples):
            if s['type'] == 'real' and s.get('ext_id') == external_id:
                idx_to_remove = i
                break
        if idx_to_remove is None:
            raise ValueError(f"Образец с внешним индексом {external_id} не найден среди реальных данных.")

        sample = self.samples[idx_to_remove]
        # Удаляем файлы
        for path in (sample['npy_path'], sample['csv_path']):
            if os.path.exists(path):
                os.remove(path)
        # Удаляем из списка
        self.samples.pop(idx_to_remove)
        print(f"Образец с внешним индексом {external_id} (имя '{sample['name']}') удалён.")

## Объединение .csv файлов в один и сохранение в корень

In [ ]:
def create_combined_file(metadata_path='metadata.xlsx'):
    csv_dir = "data/csv"
    output_file = "dataset_combined.csv"

    # Загружаем метаданные
    meta_df = pd.read_excel(metadata_path, sheet_name='Table_SI')
    meta_df = meta_df[['Sample', 'Subsample', 'Type']].copy()
    meta_df['Sample'] = meta_df['Sample'].fillna(method='ffill')
    meta_df.dropna(subset=['Sample'], inplace=True)
    meta_df['Sample'] = meta_df['Sample'].astype(int)
    meta_df['Subsample'] = meta_df['Subsample'].astype(str)

    meta_dict = {}
    fallback_dict = {}
    for _, row in meta_df.iterrows():
        sample = row['Sample']
        key = (sample, normalize_subsample(row['Subsample']))
        if key not in meta_dict:
            meta_dict[key] = row['Type']
        if sample not in fallback_dict:
            fallback_dict[sample] = row['Type']

    pressures = ['1', '2', '5', '10', '20', 'LBM']
    rows = []

    for idx, fname in enumerate(sorted(os.listdir(csv_dir)), start=1):
        if not fname.endswith('.csv'):
            continue
        name = fname[:-4]
        df = pd.read_csv(os.path.join(csv_dir, fname))
        df['pressure'] = df['pressure'].astype(str)
        perm_dict = dict(zip(df['pressure'], df['permeability']))
        perms = [perm_dict.get(p, None) for p in pressures]
        perms = [float(x) if x is not None else None for x in perms]

        # Определяем тип породы
        parts = name.split('_')
        if len(parts) >= 3:
            try:
                sample = int(parts[0])
                sub_raw = '_'.join(parts[1:-1])
                sub_norm = normalize_subsample(sub_raw)
                key = (sample, sub_norm)
                rock_type = meta_dict.get(key, fallback_dict.get(sample, "unknown"))
            except ValueError:
                rock_type = "unknown"
        else:
            rock_type = "unknown"

        rows.append([idx, name, rock_type] + perms)

    columns = ["index", "sample", "rock_type"] + [f"perm_{p}MPa" if p != 'LBM' else "perm_LBM" for p in pressures]
    pd.DataFrame(rows, columns=columns).to_csv(output_file, index=False)
    print(f"Сохранён {output_file} с {len(rows)} строками")



In [ ]:
# @title
create_combined_file()
dataset = PermeabilityDataset()
print(f'перввый куб {dataset[0][2]}')
print(f'последний куб {dataset[-1][2]}')
print(f"Размер датасета: {len(dataset)}")
if len(dataset) > 0:
    sample_vol, sample_target, sample_name = dataset[0]
    print(f"Пример volume: {sample_vol}")
    print(f"Пример target: {sample_target}")
    print(f"Пример имени образца: {sample_name}")


## Предобработка данных. Работа с пропусками

Найдём столбцы в которых есть пропуски и удалим сэмплы, о которых вообще нет данных


In [ ]:
data_fix = pd.read_csv("dataset_combined.csv")

columns_of_interest = data_fix.columns[data_fix.isna().sum() > 0]
not_enough_data = data_fix[data_fix[columns_of_interest].isna().sum(axis=1) >= 5]
wrong_data = data_fix[data_fix[columns_of_interest].map(lambda x: True if x < 0 else False).sum(axis=1) >= 1]
data_fix.head(6)

In [ ]:
inappropriate_samples = pd.concat([not_enough_data, wrong_data])
print(inappropriate_samples)
print(len(data_fix))
for ext_id in inappropriate_samples['index']:
    try:
        dataset.remove_by_external_index(ext_id)
    except ValueError as e:
        print(f"Ошибка при удалении внешнего индекса {ext_id}: {e}")
create_combined_file()

Теперь нужно понять, как заменить пропущенные значения для лучшего последующего обучения

### Гипотеза: существует линейная зависимость между давлением в мПа и значением проницаемости породы

Проверим с помощью стат критерия на основе корреляции Пирсона

In [ ]:
df = pd.read_csv('dataset_combined.csv')

pressure_cols = ['perm_1MPa', 'perm_2MPa', 'perm_5MPa', 'perm_10MPa', 'perm_20MPa']
pressure_values = np.array([1.0, 2.0, 5.0, 10.0, 20.0])

results_list = []

for idx, row in df.iterrows():
    sample_id = row['sample']

    perms = np.array([row[col] for col in pressure_cols], dtype=float)
    mask = ~np.isnan(perms)
    n_points = mask.sum()

    if n_points < 2:
        results_list.append({
            'sample': sample_id,
            'n_points': n_points,
            'pearson_r': np.nan,
            'p_value': np.nan,
            'is_significant': False
        })
        continue

    p_vals = pressure_values[mask]
    k_vals = perms[mask]

    r, p = stats.pearsonr(p_vals, k_vals)

    results_list.append({
        'sample': sample_id,
        'n_points': n_points,
        'pearson_r': r,
        'p_value': p,
        'is_significant': p < 0.05
    })

results = pd.DataFrame(results_list)

print("=" * 60)
print("РЕЗУЛЬТАТЫ КОРРЕЛЯЦИОННОГО АНАЛИЗА (ПИРСОН)")
print("=" * 60)
print(f"Всего образцов: {len(results)}")

valid = results[results['n_points'] >= 2]
print(f"Образцов с >=2 точками: {len(valid)}")
print(f"Из них со значимой корреляцией: {valid['is_significant'].sum()}")
print(f"Доля значимых: {valid['is_significant'].mean():.2%}")

print("\nРаспределение коэффициента Пирсона (r):")
print(valid['pearson_r'].describe())


Видим, что линейной зависимости нет, но прослеживается существование некоторого закона изменения проницаемости в зависимости от приложенного давления

### Гипотеза: существует степенная зависимость между давлением в мПа и значением проницаемости породы

In [ ]:
for _, i in df.sample(6).iterrows():
  perms = i[pressure_cols].values.astype(float)
  plt.plot(pressure_values, perms, 'o-')
  plt.title(f"Sample: {i['sample']}")
  plt.xlabel("Pressure (MPa)")
  plt.ylabel("Permeability")
  plt.show()

Заполним наны благодаря такой зависимости

In [ ]:

def fill_permeability_gaps(df, visualize=True, n_plots=4, random_state=42, fill_lbm=True, lbm_factor=1e-12):

    pressures = np.array([1, 2, 5, 10, 20], dtype=float)
    perm_cols = [f'perm_{int(p)}MPa' for p in pressures]

    df_filled = df.copy()
    samples_with_fills = []

    def power_law(P, a, b):
        return a * np.power(P, b)

    for idx, row in df_filled.iterrows():
        values = pd.to_numeric(row[perm_cols], errors='coerce').to_numpy(dtype=float)
        valid_mask = ~np.isnan(values)
        P_valid = pressures[valid_mask]
        k_valid = values[valid_mask]

        if len(P_valid) < 2:
            continue
        if np.any(k_valid <= 0):
            continue

        try:
            logP = np.log(P_valid)
            logK = np.log(k_valid)
            coeffs = np.polyfit(logP, logK, 1)
            b = coeffs[0]
            a = np.exp(coeffs[1])
            k_pred = power_law(pressures, a, b)

            had_fill = False
            for i, col in enumerate(perm_cols):
                if pd.isna(row[col]):
                    df_filled.at[idx, col] = k_pred[i]
                    had_fill = True
            if had_fill:
                samples_with_fills.append(idx)
        except Exception as e:
            continue

    if fill_lbm and 'perm_LBM' in df_filled.columns:
        for idx, row in df_filled.iterrows():
            if pd.isna(row['perm_LBM']):
                perm_20 = df_filled.at[idx, 'perm_20MPa']
                if not pd.isna(perm_20):
                    df_filled.at[idx, 'perm_LBM'] = perm_20 * lbm_factor
                else:
                    pass

    if visualize and samples_with_fills:
        np.random.seed(random_state)
        selected = np.random.choice(samples_with_fills,
                                    size=min(n_plots, len(samples_with_fills)),
                                    replace=False)

        for idx in selected:
            row_original = df.loc[idx]
            row_filled = df_filled.loc[idx]

            orig_vals = pd.to_numeric(row_original[perm_cols], errors='coerce').to_numpy(dtype=float)
            mask_known = ~np.isnan(orig_vals)
            P_known = pressures[mask_known]
            k_known = orig_vals[mask_known]

            filled_vals = row_filled[perm_cols].to_numpy(dtype=float)
            mask_filled = np.isnan(orig_vals)
            P_filled = pressures[mask_filled]
            k_filled = filled_vals[mask_filled]

            P_smooth = np.linspace(1, 20, 100)
            logP_known = np.log(P_known)
            logK_known = np.log(k_known)
            coeffs = np.polyfit(logP_known, logK_known, 1)
            a = np.exp(coeffs[1])
            b = coeffs[0]
            k_smooth = power_law(P_smooth, a, b)

            plt.figure(figsize=(10, 5))
            plt.plot(P_smooth, k_smooth, '--', color='gray', alpha=0.8, label='Степенная модель')
            plt.plot(P_known, k_known, 'o', color='blue', markersize=8, label='Исходные данные')
            plt.plot(P_filled, k_filled, 's', color='red', markersize=8, label='Заполненные значения')
            plt.title(f"Образец: {row_original['sample']}\n")
            plt.xlabel("Эффективное давление, МПа")
            plt.ylabel("Проницаемость (единицы исходных данных)")
            plt.grid(True, linestyle=':', alpha=0.6)
            plt.legend()

    print(f"Обработано образцов: {len(df)}. Заполнены пропуски в {len(samples_with_fills)} образцах.")
    return df_filled

filled_df = fill_permeability_gaps(df)


### Пропуски заполнены, обновим датасет

In [ ]:
filled_df.isna().sum()

In [ ]:
filled_df.to_csv("dataset_combined.csv", index=False)
print(f"Обновлённый файл dataset_combined.csv сохранён.")

csv_dir = "data/csv"
pressures = ['1', '2', '5', '10', '20', 'LBM']
col_map = {
    '1': 'perm_1MPa',
    '2': 'perm_2MPa',
    '5': 'perm_5MPa',
    '10': 'perm_10MPa',
    '20': 'perm_20MPa',
    'LBM': 'perm_LBM'
}

for _, row in filled_df.iterrows():
    sample_name = row['sample']
    csv_file = os.path.join(csv_dir, f"{sample_name}.csv")

    data = []
    for p in pressures:
        col = col_map[p]
        if col in row.index:
            val = row[col]
            if pd.notna(val):
                data.append({'pressure': p, 'permeability': val})

    if data:
        sample_df = pd.DataFrame(data)
        sample_df.to_csv(csv_file, index=False)
        print(f"Обновлён: {csv_file}")
    else:
        print(f"Пропущен (нет данных): {sample_name}")

print("Все файлы обновлены.")

dataset = PermeabilityDataset()


In [ ]:
dataset[100]

## Функции визуализации породы
Данные функции позволяют отрисовать изображение породы или её среза по трехмерному бинарному массиву данных

In [ ]:
# @title Функция визуализации
def visualize(data, file_name='rock_structure_1',
              L = [(-300, 350, -500), (128, 128, 128), (1, 0, 1.5)], T = True, decimation_factor=0.9, window_size=[635, 635]):
    data_np = np.flip(data.squeeze(), axis=0)
    shape = data_np.shape

    # Создаем сетку
    grid = pv.ImageData(dimensions=shape, spacing=(1, 1, 1), origin=(0, 0, 0))
    grid.point_data["values"] = data_np.flatten(order="F")

    # Генерация изоповерхности (максимальное количество полигонов на входе)
    surface_mesh = grid.contour([0.5], scalars="values")

    # Агрессивное уменьшение количества полигонов (Decimation)
    if decimation_factor > 0 and surface_mesh.n_cells > 0:
        surface_mesh = surface_mesh.decimate(decimation_factor)

    # Настройка визуализатора
    pv.set_jupyter_backend('static')
    plotter = pv.Plotter(off_screen=True, window_size=window_size)

    # Рисуем границы куба
    plotter.add_mesh(surface_mesh, color='grey', lighting=True) ## smooth_shading=True добавлять?
    plotter.add_mesh(pv.Cube(bounds=[0, shape[0]-1, 0, shape[1]-1, 0, shape[2]-1]),
                     style='wireframe', color='black', line_width=1, opacity=0.5) # добавляем границы

    # Настройка фона и положения камеры
    plotter.set_background('white')
    plotter.camera_position = L
    plotter.add_title(file_name, font_size=10)  # название камня

    img_path = f'/tmp/{file_name}.png'
    plotter.screenshot(img_path)
    if T:
      display(Image(img_path))

In [ ]:
# @title Функции визуализации фильтраций 3D
def visualize_color_rock(data, filtration_vol, file_name='rock_structure_colored',
                         L = [(-300, 350, -500), (128, 128, 128), (1, 0, 1.5)], T = True, decimation_factor=0.9, window_size=[635, 635]):
    data_np = np.flip(data.squeeze(), axis=0)
    shape = data_np.shape
    filt_np = np.flip(filtration_vol.squeeze(), axis=0)

    # Создаем сетку
    grid_shape = (shape[0] + 1, shape[1] + 1, shape[2] + 1)
    grid = pv.ImageData(dimensions=grid_shape, spacing=(1,1,1))

    # Записываем маску породы и значения фильтрации в данные ячеек (cell_data)
    grid.cell_data["rock_mask"] = data_np.flatten(order="F").astype(np.uint8)
    grid.cell_data["dist_value"] = filt_np.flatten(order="F")

    # Переносим данные из ячеек в точки (point_data), так как алгоритм contour работает с точками
    grid = grid.cell_data_to_point_data()

    # Создаем полигональную сетку (изоповерхность) на границе пора/порода (значение 0.5)
    mesh = grid.contour([0.5], scalars="rock_mask")

    # Уменьшаем количество полигонов для ускорения отрисовки, если задан коэффициент
    if decimation_factor > 0 and mesh.n_cells > 0:
        mesh = mesh.decimate(decimation_factor)

    # Настройка визуализатора
    pv.set_jupyter_backend('static')
    plotter = pv.Plotter(off_screen=True, window_size=window_size)

    # Добавляем сетку в визуализатор, раскрашивая по значениям фильтрации, убираем шкалу
    if "dist_value" in mesh.point_data:
        plotter.add_mesh(mesh, scalars="dist_value", cmap="jet", lighting=True, show_scalar_bar=False)
    else:
        # Если данные потерялись при децимации, интерполируем их заново
        mesh = mesh.sample(grid)
        plotter.add_mesh(mesh, scalars="dist_value", cmap="jet", lighting=True, show_scalar_bar=False)

    # Рисуем границы куба
    plotter.add_mesh(pv.Cube(bounds=[0, shape[0]-1, 0, shape[1]-1, 0, shape[2]-1]),
                     style='wireframe', color='black', opacity=0.5)

    # Настройка фона и положения камеры
    plotter.set_background('white')
    plotter.camera_position = L

    # Сохраняем скриншот и отображаем его в ноутбуке
    img_path = f'/tmp/{file_name}.png'
    plotter.screenshot(img_path)
    if T:
      display(Image(img_path))


def visualize_color_pore(data, filtration_vol, file_name='pore_cube_faces',
                         L=[(-300, 350, -500), (128, 128, 128), (1, 0, 1.5)], T = True, window_size=[635, 635]):
    data_np = np.flip(data.squeeze(), axis=0)
    shape = data_np.shape
    filt_np = np.flip(filtration_vol.squeeze(), axis=0)

    # Создаем сетку
    grid_shape = (shape[0] + 1, shape[1] + 1, shape[2] + 1)
    grid = pv.ImageData(dimensions=grid_shape, spacing=(1,1,1))

    # Если порода (data == 0), присваиваем -0.1 для серого цвета
    vis_values = np.where(data_np == 1, filt_np, -0.1)
    grid.cell_data["colors"] = vis_values.flatten(order="F")

    # Настройка визуализатора
    pv.set_jupyter_backend('static')
    plotter = pv.Plotter(off_screen=True, window_size=window_size)

    # Извлекаем только внешние грани куба
    outline = grid.outline()
    faces = grid.extract_surface(algorithm='dataset_surface')

    # Создаем модифицированный jet
    jet_colors = plt.get_cmap('jet')(np.linspace(0, 1, 256))
    new_colors = np.vstack([np.array([106/255, 104/255, 99/255, 1.0]), jet_colors])
    custom_cmap = LinearSegmentedColormap.from_list('jet_with_grey', new_colors)

    plotter.add_mesh(faces, scalars="colors", cmap=custom_cmap, clim=[-0.1, 1.0],
                     lighting=False, show_scalar_bar=False)

    # Рисуем границы куба
    plotter.add_mesh(outline, color="black", line_width=1)

    # Настройка фона и положения камеры
    plotter.set_background('white')
    plotter.camera_position = L

    # Сохраняем скриншот и отображаем его в ноутбуке
    img_path = f'/tmp/{file_name}.png'
    plotter.screenshot(img_path)
    if T:
      display(Image(img_path))

In [ ]:
# @title Функции отрисовки срезов
def show_slice(data, k=128, axis = 'z', T = True):
    if axis == 'z':
      # Извлекаем срез по оси z
      slice_2d = data.squeeze()[:, k, :]
    elif axis == 'x':
      # Извлекаем срез по оси x
      slice_2d = data.squeeze()[k, :, :]
    else:
      # Извлекаем срез по оси y
      slice_2d = data.squeeze()[:, :, k]

    if not T:
      return slice_2d

    # Визуализация среза
    plt.figure(figsize=(8, 8))
    plt.imshow(slice_2d, cmap=ListedColormap(['#6a6863', 'white'])) # 1-белый (поры), 0-серый (порода)
    plt.title(f"Срез {axis} = {k}")
    plt.axis('off')
    plt.show()


def show_slice_filtered_pore(data, normalized_grad, k=128, axis = 'z', T = True):
    # Применяем маску: там где порода (data == 0), ставим -0.1 для серого цвета
    vis_volume = np.where(data.squeeze() == 1, normalized_grad.squeeze(), -0.1)

    if axis == 'z':
      # Извлекаем срез по оси z
      slice_2d = vis_volume[:, k, :]
    elif axis == 'x':
      # Извлекаем срез по оси x
      slice_2d = vis_volume[k, :, :]
    else:
      # Извлекаем срез по оси y
      slice_2d = vis_volume[:, :, k]

    if not T:
      return slice_2d

    # Создаем копию jet, но добавляем цвет #6a6863 для породы
    base_cmap = plt.get_cmap('jet')
    new_colors = base_cmap(np.linspace(0, 1, 256))
    custom_cmap = LinearSegmentedColormap.from_list('custom_jet', new_colors)
    custom_cmap.set_under('#6a6863')

    plt.figure(figsize=(8, 8))
    plt.imshow(slice_2d, cmap=custom_cmap, vmin=0, vmax=1)
    plt.title(f"Срез {axis} = {k}")
    plt.axis('off')
    plt.show()


def show_slice_filtered_rock(data, normalized_grad, k=128, axis = 'z', T = True):
    # Применяем маску: там где поры (data == 1), ставим -0.1 для белого цвета
    vis_volume = np.where(data.squeeze() == 0, normalized_grad.squeeze(), -0.1)

    if axis == 'z':
      # Извлекаем срез по оси z
      slice_2d = vis_volume[:, k, :]
    elif axis == 'x':
      # Извлекаем срез по оси x
      slice_2d = vis_volume[k, :, :]
    else:
      # Извлекаем срез по оси y
      slice_2d = vis_volume[:, :, k]

    if not T:
      return slice_2d

    # Создаем копию jet, но для пор устанавливаем белый цвет
    base_cmap = plt.get_cmap('jet')
    new_colors = base_cmap(np.linspace(0, 1, 256))
    custom_cmap = LinearSegmentedColormap.from_list('custom_jet_rock', new_colors)
    custom_cmap.set_under('white')

    plt.figure(figsize=(8, 8))
    plt.imshow(slice_2d, cmap=custom_cmap, vmin=0, vmax=1)
    plt.title(f"Срез {axis} = {k}")
    plt.axis('off')
    plt.show()

In [ ]:
# @title Убираем временные картинки
def clear_tmp_images(extension="png"):
    """Удаляет временные изображения из /tmp/"""
    files = glob.glob(f"/tmp/*.{extension}")
    for f in files:
        try:
            os.remove(f)
        except OSError as e:
            print(f"Ошибка при удалении {f}: {e}")
    print(f"Очищено файлов: {len(files)}")

# Пример использования:
clear_tmp_images()

In [ ]:
# @title ДЛЯ ОЗУ
'''import psutil
import os
import sys
import pandas as pd

# 1. Общая статистика системы
!free -h

# 2. Поиск самых больших переменных в памяти Python
def get_size(obj, seen=None):
    size = sys.getsizeof(obj)
    if seen is None:
        seen = set()
    obj_id = id(obj)
    if obj_id in seen:
        return 0
    seen.add(obj_id)
    if isinstance(obj, dict):
        size += sum([get_size(v, seen) for v in obj.values()])
        size += sum([get_size(k, seen) for k in obj.keys()])
    elif hasattr(obj, '__dict__'):
        size += get_size(obj.__dict__, seen)
    elif hasattr(obj, '__iter__') and not isinstance(obj, (str, bytes, bytearray)):
        size += sum([get_size(i, seen) for i in obj])
    return size

# Вывод топ-10 переменных
local_vars = globals().copy()
mem_usage = []
for var_name, var_val in local_vars.items():
    if not var_name.startswith('_'):
        try:
            size = sys.getsizeof(var_val) / (1024**2) # MB
            mem_usage.append({'Variable': var_name, 'Size (MB)': size})
        except:
            pass

df_mem = pd.DataFrame(mem_usage).sort_values('Size (MB)', ascending=False).head(10)
print("\nТоп-10 переменных в памяти:")
display(df_mem)'''

In [ ]:
# @title ДЛЯ ОЗУ
'''# Проверим реальный размер в памяти для разных типов
shape = (256, 256, 256)
array_f64 = np.zeros(shape, dtype=np.float64)
array_f32 = np.zeros(shape, dtype=np.float32)

print(f"Размер float64 (default): {array_f64.nbytes / 1024**2:.1f} MB")
print(f"Размер float32: {array_f32.nbytes / 1024**2:.1f} MB")
print(f"Текущий тип массива height_filt: {height_filt.dtype}")'''

In [ ]:
# @title Пример визуализации породы
n = 100
vol_tensor, target, sample_name = dataset[n]
vol_tensor = 1 - vol_tensor
data = vol_tensor.squeeze().numpy().astype(np.uint8)

print(f'Имя файла {sample_name}, размер {data.shape}') ## выяснить из статьи нормальные названия и вытащить их
print(f'Пористость породы {int(np.sum(data)) / 256**3}') ## почему 1 у нас - это поры ???
visualize(data, sample_name)


## Функции фильтраций
Фнкции фильтрации позволяют покрасить пустоты в
 градиент, что в дальнейшем позволит делать градацию серого и считать персистентные гомологии.

In [ ]:
# @title Height filtration
def apply_height_filtration(shape, normal=(0, 0, 1), d=0):
    # Создаем сетку координат, центрированную относительно центра куба
    z, y, x = np.ogrid[:shape[0], :shape[1], :shape[2]]
    z_c, y_c, x_c = z - (shape[0] // 2 - 1), y - (shape[0] // 2 - 1), x - (shape[0] // 2 - 1)

    # Нормируем вектор нормали
    n = np.array(normal) / np.linalg.norm(normal)

    # Вычисляем расстояние от каждой точки до плоскости
    dist_from_plane = x_c * n[0] + y_c * n[1] + z_c * n[2] - d
    filtration_values = np.abs(dist_from_plane)

    # Нормировка (0.0 - 1.0) для цвета
    v_min, v_max = filtration_values.min(), filtration_values.max()
    normalized_grad = (filtration_values - v_min) / (v_max - v_min + 1e-9)

    return normalized_grad


In [ ]:
# @title Line filtration (New)
def apply_line_filtration(shape, p1=(0, 0, 0), p2=None):

    if p2 is None:
        p2 = (shape[0]-1, shape[1]-1, shape[2]-1)

    # Создаем сетку координат
    z, y, x = np.ogrid[:shape[0], :shape[1], :shape[2]]
    p1 = np.array(p1)
    p2 = np.array(p2)

    # Векторы от p1 до всех точек сетки
    dx, dy, dz = x - p1[0], y - p1[1], z - p1[2]

    # Направляющий вектор прямой (p2 - p1) и его норма
    v = p2 - p1
    v_norm_sq = np.sum(v**2) + 1e-9

    # Расстояние от точки до прямой
    cross_x = dy * v[2] - dz * v[1]
    cross_y = dz * v[0] - dx * v[2]
    cross_z = dx * v[1] - dy * v[0]

    filtration_values = np.sqrt((cross_x**2 + cross_y**2 + cross_z**2) / v_norm_sq)

    # Нормировка (0.0 - 1.0) для цвета
    v_min, v_max = filtration_values.min(), filtration_values.max()
    normalized_grad = (filtration_values - v_min) / (v_max - v_min + 1e-9)

    return normalized_grad

In [ ]:
# @title Radial filtration
def apply_radial_filtration(shape, p=None):

    if p is None:
        p = (shape[0]//2, shape[1]//2, shape[2]//2)

    # Создаем сетку координат
    z, y, x = np.ogrid[:shape[0], :shape[1], :shape[2]]

    # Вычисляем квадраты расстояний от центра p для каждой оси
    dist_sq = (x - p[0])**2 + (y - p[1])**2 + (z - p[2])**2
    filtration_values = np.sqrt(dist_sq)

    # Нормировка (0.0 - 1.0) для цвета
    v_min, v_max = filtration_values.min(), filtration_values.max()
    normalized_grad = (filtration_values - v_min) / (v_max - v_min + 1e-9)

    return normalized_grad

In [ ]:
# @title Словарь всех конфигураций фильтраций

FILTRATIONS = {
    # Имя: функция, возвращающая (1 - pore_value, rock_value)
    'radial_center': lambda shape: (apply_radial_filtration(shape, p=(shape[0]//2,)*3), 1.0),
    'radial_corner': lambda shape: (apply_radial_filtration(shape, p=(0,0,0)), 1.0),
    'radial_edge': lambda shape: (apply_radial_filtration(shape, p=(0, shape[1]//2, shape[2]//2)), 1.0),
    'height_z': lambda shape: (apply_height_filtration(shape, normal=(0,0,1)), 1.0),
    'height_x': lambda shape: (apply_height_filtration(shape, normal=(1,0,0)), 1.0),
    'height_y': lambda shape: (apply_height_filtration(shape, normal=(0,1,0)), 1.0),
    'height_diag': lambda shape: (apply_height_filtration(shape, normal=(1,1,1)), 1.0),
    'line_main': lambda shape: (apply_line_filtration(shape, (0,0,0), (shape[0]-1,)*3), 1.0),
    'line_x': lambda shape: (apply_line_filtration(shape, (0, shape[1]//2, shape[2]//2),
                                                   (shape[0]-1, shape[1]//2, shape[2]//2)), 1.0),
    'line_y': lambda shape: (apply_line_filtration(shape, (shape[0]//2, 0, shape[2]//2),
                                                   (shape[0]//2, shape[1]-1, shape[2]//2)), 1.0),
    # Инвертированные: красим породу, поры = 1.0
    'radial_center_inv': lambda shape: (1.0, apply_radial_filtration(shape, p=(shape[0]//2,)*3)),
    'height_z_inv': lambda shape: (1.0, apply_height_filtration(shape, normal=(0,0,1))),
}

print(f"Всего фильтраций: {len(FILTRATIONS)}")

In [ ]:
# @title Функция визуализации фильтрации
def show_filtration(data, filt, sample_name='rock_filtration_1', axis = 'z', levels = [0, 127, 255],
                    camera_position = [(-300, 350, -500), (128, 128, 128), (1, 0, 1.5)]):
    # Предварительная генерация 3D скриншотов
    data=data.squeeze()
    filt=filt.squeeze()
    visualize(data, sample_name, camera_position, False)
    visualize_color_rock(data, filt, f'rock_{sample_name}', camera_position, False)
    visualize_color_pore(data, filt, f'pore_{sample_name}', camera_position, False)

    # Создаем общую фигуру 4x3
    fig = plt.figure(figsize=(18, 24))

    # заголовок
    fig.text(0.15, 0.91, f"Образец: {sample_name}", fontsize=20, va='center')

    # шкала min max
    sm = plt.cm.ScalarMappable(cmap='jet', norm=plt.Normalize(vmin=0, vmax=1))
    cax = fig.add_axes([0.42, 0.90, 0.53, 0.015])
    cb = plt.colorbar(sm, cax=cax, orientation='horizontal')
    cb.set_label('Filtration values', labelpad=-75, fontsize=18)
    cb.set_ticks([0, 1])
    cb.set_ticklabels(['min', 'max'], fontsize=18)

    # 3D визуализация
    paths = [f'/tmp/{sample_name}.png', f'/tmp/rock_{sample_name}.png', f'/tmp/pore_{sample_name}.png']
    for i, path in enumerate(paths):
        ax = fig.add_subplot(4, 3, i + 1)
        img = mpimg.imread(path)
        ax.imshow(img)
        ax.axis('off')


    for i, k in enumerate(levels):
        row_idx = i + 1

        # Колонка 1: show_slice
        ax_bin = fig.add_subplot(4, 3, row_idx * 3 + 1)
        ax_bin.imshow(data[:, k, :] if axis == 'z' else (data[k, :, :] if axis == 'x' else data[:, :, k]), cmap=ListedColormap(['#6a6863', 'white']))
        ax_bin.axis('off')
        # Уменьшен отступ (x=-0.05 вместо -0.15)
        ax_bin.text(-0.05, 0.5, f"{axis} = {k}", transform=ax_bin.transAxes,
                    rotation='vertical', va='center', ha='right', fontsize=16)

        # Колонка 2: show_slice_filtered_rock
        ax_rock = fig.add_subplot(4, 3, row_idx * 3 + 2)
        vis_rock = np.where(data == 0, filt, -0.1)
        rock_cmap = LinearSegmentedColormap.from_list('rock_jet', plt.get_cmap('jet')(np.linspace(0, 1, 256)))
        rock_cmap.set_under('white')
        ax_rock.imshow(vis_rock[:, k, :] if axis == 'z' else (vis_rock[k, :, :] if axis == 'x' else vis_rock[:, :, k]), cmap=rock_cmap, vmin=0, vmax=1)
        ax_rock.axis('off')

        # Колонка 3: show_slice_filtered_pore
        ax_pore = fig.add_subplot(4, 3, row_idx * 3 + 3)
        vis_pore = np.where(data == 1, filt, -0.1)
        pore_cmap = LinearSegmentedColormap.from_list('pore_jet', plt.get_cmap('jet')(np.linspace(0, 1, 256)))
        pore_cmap.set_under('#6a6863')
        ax_pore.imshow(vis_pore[:, k, :] if axis == 'z' else (vis_pore[k, :, :] if axis == 'x' else vis_pore[:, :, k]), cmap=pore_cmap, vmin=0, vmax=1)
        ax_pore.axis('off')

    # Настраиваем отступы вручную, чтобы убрать пустоту справа
    plt.subplots_adjust(left=0.08, right=0.98, top=0.88, bottom=0.05, wspace=0.05, hspace=0.1)
    plt.show()

In [ ]:
# @title ДЛЯ ОЗУ

'''height_filt = height_filt.astype(np.float32)
line_filt = line_filt.astype(np.float32)
radial_filt = radial_filt.astype(np.float32)

# Если у вас есть исходные данные (бинарная маска 0 и 1),
# их можно перевести в uint8 (всего 1 байт на число, 16 МБ на весь куб 256^3)
data = data.astype(np.uint8)

print(f"Новый тип height_filt: {height_filt.dtype}")
print(f"Новый размер height_filt: {height_filt.nbytes / 1024**2:.1f} MB")'''

In [ ]:
# @title Пример визуализации фильтраций
# height_filt = apply_height_filtration((256, 256, 256), normal=(1, 1, 1), d=100).astype(np.float32)
# line_filt = apply_line_filtration((256, 256, 256), (127, 127, 127), (255, 255, 255)).astype(np.float32)
# radial_filt = apply_radial_filtration((256, 256, 256), (127, 127, 127)).astype(np.float32)

# visualize(data, sample_name)
# visualize_color_rock(data, height_filt, f'rock_{sample_name}')
# visualize_color_pore(data, height_filt, f'pore_{sample_name}')

# show_slice(data, 0)
# show_slice_filtered_rock(height_filt, 0)
# show_slice_filtered_pore(height_filt, 0)


# show_filtration(data, radial_filt, sample_name)

## Визуализация кубического комплекса и градаций серого
В этом разделе мы визуализуруем кубические комплексы разными способами.

In [ ]:
# @title Функции визуализации grayscale
def show_slice_grayscale(grad_data, k=40, axis = 'z', T = True):
    if axis == 'z':
      # Извлекаем срез по оси z
      grayscale_2d = grad_data.squeeze()[:, k, :]
    elif axis == 'x':
      # Извлекаем срез по оси x
      grayscale_2d = grad_data.squeeze()[k, :, :]
    else:
      # Извлекаем срез по оси y
      grayscale_2d = grad_data.squeeze()[:, :, k]

    if not T:
      return grayscale_2d

    plt.figure(figsize=(8, 8))
    plt.imshow(grayscale_2d, cmap='gray') # gray 0 -> черный, 1 -> белый
    plt.title(f"Срез {axis} = {k}")
    plt.axis('off')
    plt.show()


def visualize_grayscale_faces(grad_data, file_name='grayscale_pore_cube_faces',
                         camera_position=[(-300, 350, -500), (128, 128, 128), (1, 0, 1.5)], T = True, window_size=[635, 635]):
    grad_data_np = np.flip(grad_data.squeeze(), axis=0)
    shape = grad_data_np.shape

    # Создаем сетку
    grid_shape = (shape[0] + 1, shape[1] + 1, shape[2] + 1)
    grid = pv.ImageData(dimensions=grid_shape, spacing=(1,1,1))

    # Добавляем данные в сетку
    grid.cell_data["colors"] = grad_data_np.flatten(order="F")

    # Настройка визуализатора
    pv.set_jupyter_backend('static')
    plotter = pv.Plotter(off_screen=True, window_size=window_size)

    # Извлекаем только внешние грани куба
    outline = grid.outline()
    faces = grid.extract_surface(algorithm='dataset_surface')

    # Теперь массив 'colors' существует в данных поверхности
    plotter.add_mesh(faces, scalars="colors", cmap='gray',
                     lighting=False, show_scalar_bar=False)

    # Рисуем границы куба
    plotter.add_mesh(outline, color="black", line_width=1)

    # Настройка фона и положения камеры
    plotter.set_background('white')
    plotter.camera_position = camera_position

    # Сохраняем скриншот и отображаем его в ноутбуке
    img_path = f'/tmp/{file_name}.png'
    plotter.screenshot(img_path)
    if T:
      display(Image(img_path))

In [ ]:
# @title Функции визуализации кубического комплекса
def show_cubical_complex_slice(img_filtered, k=42, axis = 'z', num_thresholds = 10):
    if axis == 'z':
      # Извлекаем срез по оси z
      slice_2d = img_filtered.squeeze()[:, k, :]
    elif axis == 'x':
      # Извлекаем срез по оси x
      slice_2d = img_filtered.squeeze()[k, :, :]
    else:
      # Извлекаем срез по оси y
      slice_2d = img_filtered.squeeze()[:, :, k]

    v_min = np.min(slice_2d)
    v_max = np.max(slice_2d)
    fig, ax = plt.subplots(ncols=num_thresholds + 1, figsize=(num_thresholds + 1, 11), dpi=300)

    ax[0].imshow(slice_2d, cmap='gray') # gray 0 -> черный, 1 -> белый
    ax[0].set_title(f"{axis} = {k}")
    ax[0].axis('off')

    for i, t in enumerate(np.linspace(v_min, v_max, num_thresholds)): # 20 порогов от 0 до 1
        ax[i+1].axis("off")
        ax[i+1].imshow(slice_2d <= t, cmap='gray_r', vmin=0, vmax=1)
        ax[i+1].set_title(f"{t:.2f}")
    plt.tight_layout()
    plt.show()


def visualize_cubical_complex_faces(img_filtered, num_thresholds = 10, title = 'cubical_complex', title_2 = "Только поры"):
      v_min = np.min(img_filtered)
      v_max = np.max(img_filtered)
      fig, ax = plt.subplots(ncols=num_thresholds + 1, figsize=(num_thresholds + 1, 11), dpi=300)

      visualize_grayscale_faces(img_filtered, f'cubical_complex_pore_{small_sample_name}', camera_position = [(-100, 150, -200), (42, 42, 42), (1, 0, 1.5)], T = False)
      ax[0].imshow(mpimg.imread(f'/tmp/{title}.png'))
      ax[0].set_title(title_2)
      ax[0].axis('off')

      for i, t in enumerate(np.linspace(v_min, v_max, num_thresholds)):
          vis_values = np.where(img_filtered <= t, 0, 1)
          visualize_grayscale_faces(vis_values, f'{title}_{i+1}', camera_position = [(-100, 150, -200), (42, 42, 42), (1, 0, 1.5)], T = False)
          ax[i+1].imshow(mpimg.imread(f'/tmp/{title}_{i+1}.png'))
          ax[i+1].set_title(f"{t:.2f}")
          ax[i+1].axis("off")
      plt.tight_layout()
      plt.show()

In [ ]:
# @title Пример отрисовки для кубика 85 на 85 на 85
grayscale_filt = radial_filt[np.newaxis, :85, :85, :85] # значения от 0 до 1 #:85, 85:170, 171:256
grayscale_data = vol_tensor.numpy()[:, :85, :85, :85] # 0 - порода, 1 - поры
small_sample_name = f'{sample_name}_111_radial'


# Передаем уровни (levels), которые не превышают размер 85
print(grayscale_filt.shape)
print(grayscale_data.shape)
show_filtration(grayscale_data, grayscale_filt, small_sample_name, levels=[0, 40, 84], camera_position = [(-100, 150, -200), (42, 42, 42), (1, 0, 1.5)])

In [ ]:
# @title Пример отрисовки разных фильтраций серого
grayscale_only_pore = np.where(grayscale_data == 1, grayscale_filt, 1) # порода белая
grayscale_only_rock = np.where(grayscale_data == 0, grayscale_filt, 1) # поры белые
grayscale_all = np.where(grayscale_data == 1, grayscale_filt / 2, (grayscale_filt + 1) / 2) # порода 0 - 1/2, поры 1/2 - 1, то есть порода темнее

print(grayscale_only_pore.shape)
print(grayscale_only_rock.shape)
print(grayscale_all.shape)

fig, ax = plt.subplots(ncols=3, nrows=2, figsize=(10, 10), dpi=200)


visualize_grayscale_faces(grayscale_only_pore, f'pore_{small_sample_name}', camera_position = [(-100, 150, -200), (42, 42, 42), (1, 0, 1.5)], T = False)
visualize_grayscale_faces(grayscale_only_rock, f'rock_{small_sample_name}', camera_position = [(-100, 150, -200), (42, 42, 42), (1, 0, 1.5)], T = False)
visualize_grayscale_faces(grayscale_all, f'multipl_{small_sample_name}', camera_position = [(-100, 150, -200), (42, 42, 42), (1, 0, 1.5)], T = False)

ax[0, 0].imshow(mpimg.imread(f'/tmp/pore_{small_sample_name}.png'))
ax[0, 0].set_title("Только поры")
ax[0, 0].axis('off')

ax[0, 1].imshow(mpimg.imread(f'/tmp/rock_{small_sample_name}.png'))
ax[0, 1].set_title("Только порода")
ax[0, 1].axis('off')

ax[0, 2].imshow(mpimg.imread(f'/tmp/multipl_{small_sample_name}.png'))
ax[0, 2].set_title("Всё")
ax[0, 2].axis('off')

ax[1, 0].imshow(show_slice_grayscale(grayscale_only_pore, 84, T = False), cmap='gray') # gray 0 -> черный, 1 -> белый
ax[1, 0].axis('off')
ax[1, 0].set_title("Срез z = 84")

ax[1, 1].imshow(show_slice_grayscale(grayscale_only_rock, 84, T = False), cmap='gray')
ax[1, 1].axis('off')
ax[1, 1].set_title("Срез z = 84")

ax[1, 2].imshow(show_slice_grayscale(grayscale_all, 84, T = False), cmap='gray')
ax[1, 2].axis('off')
ax[1, 2].set_title("Срез z = 84")

plt.tight_layout()
plt.subplots_adjust(wspace=0.05, hspace=-0.43)
plt.show()

In [ ]:
# @title Пример визуализации кубического комплекса
show_cubical_complex_slice(grayscale_only_pore[0], 84, num_thresholds = 10)
show_cubical_complex_slice(grayscale_only_rock[0], 84, num_thresholds = 10)
show_cubical_complex_slice(grayscale_all[0], 84, num_thresholds = 20)





visualize_cubical_complex_faces(grayscale_only_pore[0], title = f'cubical_complex_pore_{small_sample_name}', title_2 = "Только поры")
visualize_cubical_complex_faces(grayscale_only_rock[0], title = f'cubical_complex_pore_{small_sample_name}', title_2 = "Только порода")
visualize_cubical_complex_faces(grayscale_all[0], 20, title = f'cubical_complex_pore_{small_sample_name}', title_2 = "Всё")




## Подсчёт и визуализация персистентных диаграмм
В этом разделе мы считаем персистентные диаграммы при помощи методов библиотеки GUDHI, giotto и crisper. Делаем сравнительный анализ эффективности работы этих библиотек

In [ ]:
# @title Вспомогательная функция приведения к размеру
def resize_cube(cube, target_size=128):
    if cube.shape[0] == target_size:
        return cube.copy()

    from scipy.ndimage import zoom
    factor = target_size / cube.shape[0]
    return zoom(cube, (factor, factor, factor), order=0)

In [ ]:
# @title Вычисление ПД через GUDHI
def compute_diagrams_gudhi(cube, filtration_name, target_size=128):
    import time
    t0 = time.time()

    cube = resize_cube(cube, target_size)
    pore_mask = (cube == 0).astype(np.float64)

    pore_val, rock_val = FILTRATIONS[filtration_name](cube.shape)
    grayscale = np.where(pore_mask == 1, pore_val, rock_val).astype(np.float64)

    cc = gd.CubicalComplex(dimensions=grayscale.shape, top_dimensional_cells=grayscale.flatten())
    diagram = cc.persistence()

    points = np.array([[dim, birth, death] for (dim, (birth, death)) in diagram])
    points[np.isinf(points[:, 2]), 2] = 1.0

    return points, float(pore_mask.mean()), time.time() - t0

In [ ]:
# @title Вычисление ПД через cripser
def compute_diagrams_cripser(cube, filtration_name, target_size=128):
    import time
    t0 = time.time()

    cube = resize_cube(cube, target_size)
    pore_mask = (cube == 0).astype(np.float64)

    pore_val, rock_val = FILTRATIONS[filtration_name](cube.shape)
    grayscale = np.where(pore_mask == 1, pore_val, rock_val).astype(np.float64)

    raw = cripser.computePH(grayscale, maxdim=2)
    points = raw[:, :3].copy()
    points[np.isinf(points[:, 1]), 1] = 1.0

    return points, float(pore_mask.mean()), time.time() - t0

In [ ]:
# @title Вычисление ПД через giotto-tda
def compute_diagrams_giotto(cube, filtration_name, target_size=128):
    import time
    t0 = time.time()

    cube = resize_cube(cube, target_size)
    pore_mask = (cube == 0).astype(np.float64)

    pore_val, rock_val = FILTRATIONS[filtration_name](cube.shape)
    grayscale = np.where(pore_mask == 1, pore_val, rock_val).astype(np.float64)

    cp = CubicalPersistence(homology_dimensions=[0, 1, 2], n_jobs=1)
    diagrams = cp.fit_transform(grayscale[np.newaxis, ...])[0]

    points = diagrams[:, [2, 0, 1]]
    points[np.isinf(points[:, 1]), 1] = 1.0

    return points, float(pore_mask.mean()), time.time() - t0

In [ ]:
# @title Универсальная функция отрисовки

def plot_diagrams_comparison(points_dict, sample_name="", porosity=None,
                             min_persistence=0.01, figsize=(18, 5)):
    n_methods = len(points_dict)
    fig, axes = plt.subplots(n_methods, 3, figsize=(15, 4 * n_methods))
    if n_methods == 1:
        axes = axes.reshape(1, -1)

    colors = ['#ef553b', '#00cc96', '#ab63fa']
    dim_names = ['H0', 'H1', 'H2']

    for row, (method_name, points) in enumerate(points_dict.items()):
        # Защита: обрезаем все значения до разумного диапазона
        b = np.clip(points[:, 1], -1e10, 1e10)
        d = np.clip(points[:, 2], -1e10, 1e10)

        # Безопасный максимум
        finite = np.isfinite(b) & np.isfinite(d) & (b < 1e10) & (d < 1e10)
        if finite.any():
            gmax = float(max(b[finite].max(), d[finite].max())) * 1.05
            gmax = min(gmax, 1e10)  # дополнительная страховка
        else:
            gmax = 1.0

        for dim in range(3):
            ax = axes[row, dim]
            mask = points[:, 0] == dim
            birth = b[mask]
            death = d[mask]
            n = len(birth)

            if n > 0:
                # Заменяем inf на gmax
                death_plot = np.where(np.isinf(death) | np.isnan(death) | (death > 1e10), gmax * 1.05, death)
                birth_plot = np.where(np.isinf(birth) | np.isnan(birth) | (birth > 1e10), gmax, birth)
                ax.scatter(birth_plot, death_plot, c=colors[dim], alpha=0.6, s=8,
                          edgecolors='white', linewidth=0.2)

            ax.plot([0, gmax], [0, gmax], 'k--', alpha=0.2, linewidth=0.8)
            ax.set_xlim(0, gmax * 1.05)
            ax.set_ylim(0, gmax * 1.1)
            ax.set_aspect('equal')
            ax.set_xticks([])
            ax.set_yticks([])

            if row == 0:
                ax.set_title(f'{dim_names[dim]} ({n})', fontsize=10)
            if dim == 0:
                ax.set_ylabel(method_name, fontsize=10, rotation=0, ha='right', va='center')

    title = sample_name
    if porosity is not None:
        title += f' | Пористость: {porosity:.3f}'
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# @title Сравнение GUDHI vs cripser vs giotto

sample_idx = 0
filt_name = 'radial_center'

vol_tensor, target, sample_name = dataset[sample_idx]
cube = vol_tensor.squeeze().numpy()

for name, func in [('GUDHI', compute_diagrams_gudhi),
                   ('cripser', compute_diagrams_cripser),
                   ('giotto', compute_diagrams_giotto)]:
    points, porosity, elapsed = func(cube, filt_name)
    print(f"{name}: {elapsed:.1f}с | H0={(points[:,0]==0).sum()} H1={(points[:,0]==1).sum()} H2={(points[:,0]==2).sum()}")
    plot_diagrams_comparison({name: points}, f"{sample_name} | {name}", porosity)

In [ ]:
# @title Обработка всего датасета с помощью crisper
output_dir = "data/persistent_diagrams_all"
os.makedirs(output_dir, exist_ok=True)

# Считаем общее количество задач
total_tasks = len(dataset) * len(FILTRATIONS)

# Один общий прогресс-бар
with tqdm(total=total_tasks, desc="Вычисление ПД", unit="образец") as pbar:
    for idx in range(len(dataset)):
        vol_tensor, target, sample_name = dataset[idx]
        cube = vol_tensor.squeeze().numpy()

        for filt_name in FILTRATIONS:
            output_path = os.path.join(output_dir, f"{sample_name}_{filt_name}_pd.pkl")

            if os.path.exists(output_path):
                pbar.update(1)
                continue

            points, porosity, elapsed = compute_diagrams_cripser(cube, filt_name, target_size=128)

            with open(output_path, 'wb') as f:
                pickle.dump({
                    'sample': sample_name,
                    'filtration': filt_name,
                    'diagrams': points,
                    'target': target.numpy(),
                    'porosity': porosity
                }, f)

            # Обновляем описание с текущим образцом и фильтрацией
            pbar.set_postfix_str(f"{sample_name} | {filt_name} | {elapsed:.1f}s")
            pbar.update(1)

print("Готово!")

In [ ]:
# @title Проверка результата на одном из образцов

sample = "135_00_256"

for filt_name in FILTRATIONS:
    pkl_path = f"data/persistent_diagrams_all/{sample}_{filt_name}_pd.pkl"

    if os.path.exists(pkl_path):
        with open(pkl_path, 'rb') as f:
            data = pickle.load(f)

        plot_diagrams_comparison(
            {filt_name: data['diagrams']},
            sample_name=f"{sample} | {filt_name}",
            porosity=data.get('porosity'),
            figsize=(15, 4)
        )

In [ ]:
# @title Выгрузка данных во внешний архив
import zipfile
import os
from google.colab import files

# Что архивируем
source_dir = "/content/data/persistent_diagrams_all"  # ваша папка
output_zip = "/content/diagrams_all.zip"

# Создаём ZIP
with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, filenames in os.walk(source_dir):
        for filename in filenames:
            file_path = os.path.join(root, filename)
            arcname = os.path.relpath(file_path, os.path.dirname(source_dir))
            zipf.write(file_path, arcname)

print(f"Архив создан: {output_zip}")
print(f"Размер: {os.path.getsize(output_zip) / 1024**2:.1f} MB")

# Скачиваем
files.download(output_zip)

## Векторизация персистентных диаграмм